# CDC-2 — Debezium connector bring-up

**Break → Detect → Fix → Prove.** [CDC-1](../postgres_setup/) created a slot by hand; in production
**Kafka Connect** runs the **Debezium Postgres connector**, which creates the publication + slot,
takes a consistent **initial snapshot**, then **streams** every change. You drive it entirely
through the Connect **REST API**. By the end, row changes in Postgres show up as JSON events in Kafka.

**Prerequisite:** `make cdc-up`. **Laptop-safe:** 20-row table, bounded topic reads, full teardown.

In [ ]:
from common import cdc_helpers as cdc
import json, time

print("Connect REST up:", cdc.connect_up(timeout=40))
cdc.teardown("cdc2-orders", "cdc2_orders")     # clean slate (idempotent re-run)
cdc.seed_orders("cdc2_orders", n=20)
TOPIC = cdc.topic_name("cdc2_orders")
print("seeded public.cdc2_orders; Debezium will publish to topic:", TOPIC)

## 1. The connector config

The key fields (`common.cdc_helpers.debezium_pg_config` fills sensible defaults):

- `connector.class` — the Debezium Postgres connector
- `plugin.name=pgoutput` — Postgres 16's built-in logical-decoding output plugin
- `slot.name` / `topic.prefix` — the replication slot and the `<prefix>.<schema>.<table>` topic name
- `table.include.list` — which tables to capture
- `snapshot.mode=initial` — snapshot the table once, then stream (see CDC-3 for other modes)
- `decimal.handling.mode=double` — emit NUMERIC as a readable number, not base64 bytes

In [ ]:
cfg = cdc.debezium_pg_config("cdc2-orders", "cdc2_orders", snapshot_mode="initial")
print(json.dumps(cfg, indent=2))

## 2. Register it (PUT to the Connect REST API)

In [ ]:
reg = cdc.register_connector("cdc2-orders", cfg)
print("register -> HTTP", reg["status"])             # 201 created
print("state    ->", cdc.wait_for_connector("cdc2-orders", timeout=60))  # RUNNING

## 3. Detect — connector status + the auto-created slot

Debezium created the replication slot for us; unlike CDC-1's hand-made slot, this one is
**active** (a consumer — the connector — is attached and advancing it).

In [ ]:
st = cdc.connector_status("cdc2-orders")
print("connector:", st["connector"]["state"], "| tasks:", [t["state"] for t in st["tasks"]])
for s in cdc.list_slots():
    print("slot:", s)                                # cdc2_orders_slot, active=True

## 4. The initial snapshot — READ events

With `snapshot.mode=initial`, the connector first reads the whole table and emits **one `r` (read)
event per row**. `before=null`, `after={the row}`, `op="r"`.

In [ ]:
ev = cdc.read_cdc_events(TOPIC)
print("op counts:", cdc.op_counts(ev))               # {'r': 20}
print("\none snapshot event:")
print(json.dumps({k: ev[0]["raw"][k] for k in ("before", "after", "op")}, indent=2))

## 5. Streaming — a live change becomes a `c` event

Now insert a row directly in Postgres. The connector decodes it from the WAL and emits a
**create (`c`)** event onto the same topic — no snapshot, no polling.

In [ ]:
cdc.pg_exec("INSERT INTO public.cdc2_orders(id,customer,amount,status) VALUES (100,'live',9.90,'NEW')")
time.sleep(5)                                        # streaming has a small decode lag
ev = cdc.read_cdc_events(TOPIC)
print("op counts now:", cdc.op_counts(ev))           # {'r': 20, 'c': 1}
new = [e for e in ev if e["op"] == "c"][0]
print("create event after:", new["after"])

## 6. Prove & Diagnose

- **Snapshot then stream, one topic.** `r` events (the 20-row snapshot) and the `c` event (the live
  insert) all land on `dbz.public.cdc2_orders`. Browse it live in **kafka-ui** →
  http://localhost:8080 → topic `dbz.public.cdc2_orders`.
- The connector is now a **healthy consumer** of the slot — contrast CDC-1's inactive slot.

## 7. Takeaways & "in real production…"

- Connectors are **declarative + REST-managed**: `PUT /connectors/<n>/config` is idempotent, so
  config lives in version control and is reapplied, not click-opsed.
- `snapshot.mode` decides startup behavior (CDC-3); the **event envelope** (`before`/`after`/`op`/
  `ts_ms`) is the contract you build on (CDC-4).
- Watch connector/task **state** and **lag**; a `FAILED` task or an inactive slot is your page.

## Teardown

In [ ]:
cdc.teardown("cdc2-orders", "cdc2_orders")    # delete connector, drop slot, drop table, delete topic
print("slots now:", cdc.list_slots())